# GP1 ASR — Local (CRDNN)

Run this notebook locally to perform a random HP search for the CRDNN model and generate a
competition submission.

**Setup:** The `.venv` virtual environment must be set up via `uv sync` beforehand (select it as
the Jupyter kernel). Place your data in `data/train/`, `data/dev/`, and `data/test/` under the
repo root (`group-projects/gp1/`), each subfolder containing the CSV manifest and wav files.
No additional installs or cloning are needed — everything runs from the local checkout. The
notebook auto-detects `GP1_ROOT` by walking up from the notebook directory. Runs `N_TRIALS`
training trials and writes `submission.csv` to the repo root.


In [ ]:
import subprocess, sys
from pathlib import Path
import torch

# Assume notebook sits in notebooks/, repo root is the parent of parent
GP1_ROOT = Path.cwd().resolve()
while not (GP1_ROOT / "scripts" / "train.py").exists():
    if GP1_ROOT == GP1_ROOT.parent:
        raise RuntimeError("Could not locate scripts/train.py — cd to group-projects/gp1")
    GP1_ROOT = GP1_ROOT.parent
sys.path.insert(0, str(GP1_ROOT / "src"))

DEVICE = "mps" if torch.backends.mps.is_available() else (
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("GP1_ROOT:", GP1_ROOT)
print("device:", DEVICE)


## 1. Data paths

In [ ]:
DATA_ROOT = GP1_ROOT / "data"
TRAIN_CSV      = DATA_ROOT / "train" / "train.csv"
TRAIN_WAV_ROOT = DATA_ROOT / "train"
DEV_CSV        = DATA_ROOT / "dev"   / "dev.csv"
DEV_WAV_ROOT   = DATA_ROOT / "dev"
TEST_CSV        = DATA_ROOT / "test"  / "test.csv"
TEST_WAV_ROOT  = DATA_ROOT / "test"
for p in [TRAIN_CSV, DEV_CSV, TEST_CSV]:
    assert p.exists(), f"Missing: {p}"
print("DATA_ROOT:", DATA_ROOT)


## 1b. Precompute audio cache (one-shot)

Run `scripts/precompute_audio.py` to decode + resample train/dev to 16 kHz PCM_16 WAVs under a cache dir. Idempotent — re-running skips existing files. Saves ~40-60% per-epoch dataloading time after the first run.

Test/ is NOT precomputed — it is read once for the final submission.

In [ ]:
TRAIN_CACHE_DIR = DATA_ROOT / "_cache" / "train"
DEV_CACHE_DIR   = DATA_ROOT / "_cache" / "dev"

for csv, root, cache in [
    (TRAIN_CSV, TRAIN_WAV_ROOT, TRAIN_CACHE_DIR),
    (DEV_CSV,   DEV_WAV_ROOT,   DEV_CACHE_DIR),
]:
    cache.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [sys.executable, "scripts/precompute_audio.py",
         "--csv", str(csv), "--root", str(root),
         "--output-dir", str(cache),
         "--num-workers", "4"],
        cwd=str(GP1_ROOT), check=True,
    )
print("train cache:", TRAIN_CACHE_DIR)
print("dev   cache:", DEV_CACHE_DIR)


## 2. HP-search config + helpers

In [ ]:
BASELINE = "crdnn"
CONFIG_NAME = "crdnn.yaml"
# local is slower, smaller budget
N_TRIALS = 10
RANDOM_SEED = 42
NUM_WORKERS = 4
OUTPUT_BASE = GP1_ROOT / "_local_runs" / BASELINE
SUBMISSION = GP1_ROOT / "submission.csv"

import math, json, yaml, random
from gp1.config import load_config as _load_config


def sample_params(rng: random.Random) -> dict:
    return {
        "lr": 10 ** rng.uniform(-4, math.log10(5e-3)),
        "dropout": rng.uniform(0.05, 0.25),
        "warmup_steps": rng.randint(500, 8000),
        "grad_clip_norm": rng.choice([0.5, 1.0, 2.0, 5.0]),
        "specaug_time_mask_param": rng.choice([15, 25, 35, 50]),
    }


def patch_config(base_path: Path, params: dict, out_path: Path) -> None:
    cfg = _load_config(base_path)
    cfg.setdefault("train", {})
    cfg["train"]["lr"] = params["lr"]
    cfg["train"].setdefault("optimizer", {})["lr"] = params["lr"]
    cfg["train"].setdefault("scheduler", {})["warmup_steps"] = params["warmup_steps"]
    cfg["train"]["grad_clip_norm"] = params["grad_clip_norm"]
    cfg.setdefault("model", {})["dropout"] = params["dropout"]
    cfg.setdefault("aug", {})["specaug_time_mask_param"] = params["specaug_time_mask_param"]
    with open(out_path, "w") as f:
        yaml.safe_dump(cfg, f)


## 3. Run random-search trials

In [ ]:
rng = random.Random(RANDOM_SEED)
trials_dir = OUTPUT_BASE / "hp_search"
trials_dir.mkdir(parents=True, exist_ok=True)
trials_log = trials_dir / "trials.jsonl"
best = {"trial_id": None, "cer": float("inf"), "ckpt": None, "params": None}

for trial_id in range(N_TRIALS):
    params = sample_params(rng)
    trial_dir = trials_dir / f"trial_{trial_id:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    patched = trial_dir / "config.yaml"
    patch_config(GP1_ROOT / "configs" / CONFIG_NAME, params, patched)
    cmd = [
        sys.executable, "scripts/train.py",
        "--config", str(patched),
        "--train-csv", str(TRAIN_CSV), "--train-root", str(TRAIN_WAV_ROOT),
        "--dev-csv",   str(DEV_CSV),   "--dev-root",   str(DEV_WAV_ROOT),
        "--output-dir", str(trial_dir),
        "--num-workers", str(NUM_WORKERS),
        "--audio-cache-dir", str(TRAIN_CACHE_DIR),
        "--device", DEVICE,
    ]
    proc = subprocess.run(cmd, cwd=str(GP1_ROOT), capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"\ntrial {trial_id:03d}: FAILED (rc={proc.returncode})")
        print("--- stderr (last 25 lines) ---")
        print("\n".join(proc.stderr.splitlines()[-25:]))
        (trial_dir / "stderr.log").write_text(proc.stderr)
        cer = float("inf"); result = {"error": f"rc={proc.returncode}"}
    else:
        try:
            result = json.loads((trial_dir / "result.json").read_text())
            cer = float(result["best_val_cer"])
        except Exception as exc:
            print(f"\ntrial {trial_id:03d}: could not parse result.json: {exc}")
            cer = float("inf"); result = {"error": str(exc)}
    with open(trials_log, "a") as f:
        f.write(json.dumps({"trial_id": trial_id, "params": params, "best_val_cer": cer}) + "\n")
    if cer < best["cer"]:
        best = {
            "trial_id": trial_id,
            "cer": cer,
            "ckpt": result.get("best_ckpt_path") if isinstance(result, dict) else None,
            "params": params,
        }
    print(f"trial {trial_id:03d}: CER={cer:.4f}  lr={params['lr']:.2e}")

print(f"\nBEST trial_id={best['trial_id']} CER={best['cer']:.4f}  ckpt={best['ckpt']}")
(trials_dir / "best_trial.json").write_text(json.dumps(best, indent=2, default=str))


## 4. Predict best → submission.csv

In [ ]:
assert best["ckpt"] is not None, "All trials failed — cannot produce submission"

subprocess.run(
    [sys.executable, "scripts/predict.py",
     "--checkpoint", best["ckpt"],
     "--config", str(trials_dir / f"trial_{{best['trial_id']:03d}}" / "config.yaml"),
     "--test-csv", str(TEST_CSV), "--test-root", str(TEST_WAV_ROOT),
     "--output", str(SUBMISSION),
     "--device", DEVICE],
    cwd=str(GP1_ROOT), check=True,
)

import pandas as pd
print("Submission:", SUBMISSION)
print(pd.read_csv(SUBMISSION).head(10))
